In [406]:
import numpy as np
import pandas as pd

class topo2rest():
   def __init__(self, ifile:str, temps:list = [300.0, 500.0], nreps:int = 20, kappa:float = 1.00):
      '''Convert processed topology to REST2/3 input topology
         E_tot = gamma*E^{pp} + sqrt(gamma)*E^{pw} + E^{ww}
         gamma = T_0/T_i
         for REST2/3 bonds and angles are not scaled per the REST2 paper,
         however, for REST2 LJ parameters epsilon_i is scaled by epsilon_i*gamma
         for tempered atoms and all others are unmodified. 
         In the case of REST3, with the additon of sqrt(gamma)*kappa*E^{pw} which differs 
         from sqrt(gamma)*E^{pw}, we produced the combination rule 2 nonbonded terms
         involving protein-water interactions to override the pw nonbonded interactions
         as including gamma*kappa with each hot atom epsilon_i would result in the 
         incorrect form of: E_tot = gamma*kappa*E^{pp} + sqrt(gamma*kappa)*E^{pw} + E^{ww}:
         rather than the correct form: 
         E_tot = gamma*kappa*E^{pp} + sqrt(gamma)*kappa*E^{pw} + E^{ww}
         input
         ifile = inputtopology.top
         temps = ["lower temp":float, "upper temp":float ]; temperature range of replicas
         kappa:float = kappa scaling; if not equal to 1 REST3 implementation active
         hot_m = hot molecule; [0] for most systems will select the protein'''

      self.nreps = nreps
      self.kappa = kappa
      self.hot_m = None
      self.tempreps = self.compute_temperatures(temps)
      self.lambdai = self.compute_lambda()
      self.sections = {}
      self.sections_out = {}
      self.scaled_dihedrals = {}
      self.scaled_dihedral_types = {}
      self.hard_order_sections = [ "defaults", "atomtypes", "nonbond params", "bondtypes", \
                                   "constrainttypes", "angletypes", "dihedraltypes" , "moleculetype"]
      with open(ifile) as topo:
         self.readfile = topo.readlines()
      self.gather_param_sections()
         
   def compute_lambda(self):
      return [ Ti/self.tempreps[0] for Ti in self.tempreps ]
   
   def compute_temperatures(self, temp_range:list):
      from numpy import log, exp
      tlow, thigh = temp_range
      temps = []
      for i in range(self.nreps):
         temps.append(tlow*exp((i)*log(thigh/tlow)/(self.nreps-1)))
      return temps
   
   def parse_section(self,trunks):
      first_round = True
      output = []
      for line in self.readfile[trunks:]:
         if "[" not in line and first_round!=True and len(line.split()) != 0:
            output.append(line)
         elif ";" == line[0]: 
            #output.append(line)
            continue
         elif "[" in line and first_round!=True: 
            break
         first_round = False
      return output

   def get_molecule_atomtypes(self,molecule:int=0):
      import pandas as pd
      b=[]
      for i in self.sections['moleculetype'][molecule]['atoms']:
         if len(i.split())>1 and ';' not in i.split()[0] and '[' not in i.split()[0] :
            b.append(i.split()[:2])
      dataset = np.array(b,dtype=object)
      print(dataset)   
      # Need to grab unique atoms
      atomtypes = dataset[:,1]
      return np.unique(atomtypes)
   
   def get_scale_nonbonded(self):
      import pandas as pd
      for hot in self.hot_m:
         nbp_out = []
         atom_es = self.get_molecule_atomtypes(hot)
         for nbp in self.sections['nonbond params']:
            if '[' in nbp:
               continue
            elif ';' in nbp[:3]:
               nbp_out.append(nbp)
            elif '\n' in nbp[:3]:
               continue
            else:
               nbp_ = nbp.split()
               stringout = f'{nbp_[0]:<12} {nbp_[1]:<6} {nbp_[2]:>6.3f} {nbp_[3]:>8.4f}{nbp_[4]:^5d}{nbp_[5]:>11.5e}{nbp_[6]:>13.5e}'
            self.sections_out['nonbond params']
      # TODO everything
   
#   def show_molecule_names(self):
#      try:
#         pass
         

   def get_molecule_atoms(self):
      import pandas as pd
      b=[]
      for i in self.sections['atoms']:
         i.split()
         if len(i.split())>1 and i.split()[0]!=';' and i.split()[0]!='[' :
            b.append(i.split())
   
   def _moleculetype_sub(self,linestart:int):
      first_round = True
      output = []
      for line in self.readfile[linestart:]:
         if "[" not in line and ';' not in line[:3]:
            output.append(line)
         elif ";" == line[0]:
            #output.append(line)
            continue
         elif "[" in line and first_round!=True: 
            break
         first_round = False
      return output 
   
   def identify_moltype_sections(self,trunks:int):
      section_start = []
      for i, line in enumerate(self.readfile[trunks:]):
         if '[' in line and 'moleculetype' not in line and 'system' not in line:
            section_start.append(i+trunks)
         elif i != 0 and 'moleculetype' in line or 'system' in line:
            break
      return section_start

   def parse_moleculetypes(self,trunks:int):
      first_round = True
      output = {}
      sections = self.identify_moltype_sections(trunks)
      output['header'] = self.readfile[trunks:trunks+2]
      for section in sections:
         section_ = self.readfile[section].split()[1]
         output[section_] = self._moleculetype_sub(section)
      return output
   
   def get_scale_dehedrals_(self, hot_m:list = [0]):
      # need to get atom reference for which parameters to scale 
#      dihedraltypes = pd.DataFrame([i.split()[:-2] if i.split()[-2] == ';' else i.split()[:-1] if i.split()[-1] == ';' \
#                               else i.split() for i in self.sections['dihedraltypes'] if ';' not in i[:3] if '[' not in i[:3] \
#                               if '\n' not in i[:3]], columns=['i','j','k','l','func','phase','K','mult'])
#      print(dihedraltypes)
      
      dihedrals_new = {}
      dihedral_types_new = {}
      dihedrals = self.sections['moleculetype'][hot]['dihedrals']
      dihedral_types = [" ".join(i.split()[:-2]) if i.split()[-2] == ';' else i.split()[:-1] if i.split()[-1] == ';' \
                               else i.split() for i in self.sections['dihedraltypes'] if ';' not in i[:3] if '[' not in i[:3] \
                               if '\n' not in i[:3]]
      for hot in hot_m:
         for lambdai in self.lambdai:
            dih_new = []
            dih_types_new = []
            for dihedral in dihedrals:
               dls_ = dihedral.split()
               if len(dls_) == 5:
                  dih_new.append(dihedral)
               elif len(dls_) == 8:
                  Kscaled = float(dls_[6])*lambdai
                  stringout = f'{dls_[0]:>5d} {dls_[1]:>5d} {dls_[2]:>5d} {dls_[3]:>5d} {dls_[4]:^9d}{dls_[6]:<10.5f}{Kscaled:<10.3f}{dls_[7]}\n'
                  dih_new.append(stringout)
               else: print("Warning: incorrect parsing of dihedrals section\n expecting 5 or 8 columns\n{line}")
            for dihedraltype in dihedral_types:
               dtls_ = dihedraltype.split()
               Kscaled = float(dtls_[6])*lambdai
               if len(dtls_) == 8:
                  dih_types_new.append(dihedraltype)
                  if dtls_[0] !='X' and dtls_[3] != 'X':
                     stringout = f'{"s"+dtls_[0]:>5d} {"s"+dtls_[1]:>5d} {"s"+dtls_[2]:>5d} {"s"+dtls_[3]:>5d} {dtls_[4]:^9d}{dtls_[6]:<10.5f}{Kscaled:<10.3f}{dtls_[7]}\n'
                     dih_types_new.append(stringout)
                  elif dtls_[3] == 'X':
                     if dtls_[0] == 'X':
                        stringout = f'{dtls_[0]:>5d} {"s"+dtls_[1]:>5d} {"s"+dtls_[2]:>5d} {dtls_[3]:>5d} {dtls_[4]:^9d}{Kscaled:<10.3f}{dtls_[6]:<10.5f}{dtls_[7]}\n' 
                        dih_types_new.append(stringout)
                  else: print(f'warning: parameter not found for {dtls_[:4]}')
               else: print('warning: dihedraltype not processed:\n '+dihedraltype)
                     
            dihedrals_new[lambdai] = dih_new
            dihedral_types_new[lambdai] = dih_types_new
            
         self.scaled_dihedrals[hot] = dihedrals_new
         self.scaled_dihedral_types[hot] = dihedral_types_new

   def gather_param_sections(self):
      for section in self.hard_order_sections[:-1]:
         is_select = [ i for i, line in enumerate(self.readfile) if section in line ]
         in_select = [f' [ {section} ] \n']
         for i in is_select:
            in_select += self.parse_section(i)
         self.sections[section]=in_select
      section = self.hard_order_sections[-1]
      is_select = [ i for i, line in enumerate(self.readfile) if section in line ] 
      moltype_dict = {} 
      for i in range(len(is_select)):
         moltype_dict[i] = self.parse_moleculetypes(is_select[i])
      self.sections[section] = moltype_dict
         

In [407]:
test = topo2rest('./example_topo/processed.top')

In [410]:
test.sections['nonbond params']

[' [ nonbond params ] \n',
 '; i    j    funct    sigma   epsilon\n',
 '  OB   HB     1      0.150   1.2552\n']

In [157]:
f=open('test_dihedraltypes.txt','w')
f.writelines(test.sections['dihedraltypes'])
f.close()
